# **Nhóm 1: Phân Tích Sự Phụ Thuộc Vào Khuyến Mãi (Promotion Dependency)**
**Dự án**: Phân tích hành vi người dùng DAZONE  
**Mục tiêu**: Tìm hiểu sâu sắc liệu người dùng đến với dịch vụ vì giá trị cốt lõi của sản phẩm hay chỉ vì săn đón các chương trình khuyến mãi/voucher.

Nội dung phân tích gồm 3 phần chính:
1. **Tỷ lệ giao dịch tự nhiên (Organic Transactions)** sau giao dịch đầu tiên (NPU).
2. **Đóng góp dòng tiền thực (Real User Charge)** và tỷ suất lợi nhuận (ROI) trong vòng đời 6 tháng của người dùng.
3. **Hành vi bào khuyến mãi (Deal Hunter)** để định vị tệp khách hàng chỉ xuất hiện khi có khuyến mãi và biến mất khi hết chương trình.

In [ ]:
# ── Imports & Setup ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('../data/processed/result_without_user_no_promotion.csv')
df['reqDate'] = pd.to_datetime(df['reqDate'])
print(f"Số lượng giao dịch thành công và hợp lệ: {len(df):,}")
print(f"Số lượng người dùng duy nhất: {df['userID'].nunique():,}")
df.head(3)

## **Phần 1: Tỷ lệ giao dịch tự nhiên (Organic Transactions)**
**Mục tiêu**: Đo lường hành vi lặp lại giao dịch tự nhiên (không sử dụng khuyến mãi) sau khi người dùng thực hiện giao dịch đầu tiên (New Paying User - NPU).

In [3]:
# Sắp xếp dữ liệu theo userID và thời gian giao dịch để tìm giao dịch đầu tiên (NPU)
df = df.sort_values(by=['userID', 'reqDate'])
npu_df = df.groupby('userID').first().reset_index()
npu_df = npu_df[['userID', 'transID', 'campaignID', 'reqDate']].rename(
    columns={'transID': 'npu_transID', 'campaignID': 'npu_campaignID', 'reqDate': 'npu_reqDate'}
)

# Ghép thông tin NPU ngược lại bảng chính
df = df.merge(npu_df, on='userID', how='left')

# Tách các giao dịch phát sinh sau lần giao dịch NPU
subseq_df = df[df['transID'] != df['npu_transID']].copy()

total_subseq = len(subseq_df)
organic_subseq = subseq_df[subseq_df['promotion_type'] == 'unknown']
total_organic_subseq = len(organic_subseq)

pct_organic_of_subseq = (total_organic_subseq / total_subseq) * 100 if total_subseq > 0 else 0
pct_organic_of_all = (total_organic_subseq / len(df)) * 100

print(f"Tổng số giao dịch phát sinh sau NPU: {total_subseq:,}")
print(f"Số giao dịch tự nhiên (Organic) sau NPU: {total_organic_subseq:,}")
print(f"Tỷ lệ giao dịch tự nhiên trên tổng số giao dịch sau NPU: {pct_organic_of_subseq:.2f}%")
print(f"Tỷ lệ giao dịch tự nhiên sau NPU trên tổng số giao dịch toàn hệ thống: {pct_organic_of_all:.2f}%")

Tổng số giao dịch phát sinh sau NPU: 238,118
Số giao dịch tự nhiên (Organic) sau NPU: 170,075
Tỷ lệ giao dịch tự nhiên trên tổng số giao dịch sau NPU: 71.42%
Tỷ lệ giao dịch tự nhiên sau NPU trên tổng số giao dịch toàn hệ thống: 62.58%


In [4]:
# Phân tích theo Chiến dịch thu hút khách hàng (NPU Campaign)
# Lọc các chiến dịch có tối thiểu 100 giao dịch sau NPU để đảm bảo tính đại diện thống kê
camp_stats = subseq_df.groupby('npu_campaignID').agg(
    total_subseq=('transID', 'count'),
    organic_subseq=('transID', lambda x: (df.loc[x.index, 'promotion_type'] == 'unknown').sum())
).reset_index()

camp_stats['pct_organic'] = (camp_stats['organic_subseq'] / camp_stats['total_subseq']) * 100

# Loại bỏ npu_campaignID == 0 (không có campaign NPU, tức là tự nhiên đến)
camp_stats_filtered = camp_stats[(camp_stats['npu_campaignID'] != '0') & (camp_stats['npu_campaignID'] != 0) & (camp_stats['total_subseq'] >= 100)].copy()
top_camps = camp_stats_filtered.sort_values(by='pct_organic', ascending=False).head(10)

print("Top 10 Chiến dịch NPU có tỷ lệ giao dịch tự nhiên sau đó cao nhất (Volume >= 100):")
display(top_camps)

Top 10 Chiến dịch NPU có tỷ lệ giao dịch tự nhiên sau đó cao nhất (Volume >= 100):


,npu_campaignID,total_subseq,organic_subseq,pct_organic
269,9427,100,99,99.00
141,8697,101,99,98.02
224,9158,377,369,97.88
165,8901,315,307,97.46
138,8693,185,178,96.22
352,10193,288,276,95.83
263,9395,128,122,95.31
342,10071,2681,2539,94.70
270,9445,296,279,94.26
343,10072,610,574,94.10


In [5]:
# Trực quan hóa tỷ lệ giao dịch tự nhiên của các chiến dịch NPU hàng đầu bằng Plotly
fig = px.bar(
    top_camps.assign(npu_campaignID=top_camps['npu_campaignID'].astype(str)),
    x='npu_campaignID',
    y='pct_organic',
    text='pct_organic',
    color='pct_organic',
    color_continuous_scale='Viridis',
    labels={'npu_campaignID': 'Campaign ID (Chiến dịch NPU)', 'pct_organic': 'Tỷ lệ giao dịch tự nhiên (%)'},
    title='Top 10 chiến dịch NPU có tỷ lệ giao dịch tự nhiên sau đó cao nhất'
)

fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='outside',
    marker_line_width=0,
    hovertemplate='<b>Campaign ID</b>: %{x}<br><b>Tỷ lệ tự nhiên</b>: %{y:.2f}%<extra></extra>'
)

fig.update_layout(
    title_x=0.5,
    title_font=dict(size=16, family="Inter, sans-serif", color='#2c3e50'),
    font=dict(family="Inter, sans-serif", color='#2c3e50'),
    plot_bgcolor='#ffffff',
    paper_bgcolor='#ffffff',
    coloraxis_showscale=False,
    yaxis=dict(
        range=[0, 110],
        gridcolor='rgba(200, 200, 200, 0.15)',
        zerolinecolor='rgba(200, 200, 200, 0.15)'
    ),
    xaxis=dict(type='category'),
    margin=dict(t=60, b=40, l=40, r=40),
    height=450
)

fig.show()

**Kết luận**:
   - Có tới **71.42%** các giao dịch phát sinh sau lần giao dịch đầu tiên (NPU) là giao dịch tự nhiên (Organic - không cần app trợ giá). Điều này chứng minh rằng **người dùng ở lại vì giá trị thực tế của sản phẩm/dịch vụ** nhiều hơn là vì voucher.
   - Chiến dịch `9427` và `8697` là những chiến dịch có chất lượng tuyển dụng người dùng tự nhiên tốt nhất khi ghi nhận tỷ lệ giao dịch tự nhiên sau đó đạt từ **98% đến 99%**.

## **Phần 2: Đóng góp dòng tiền thực (Real User Charge) & Vòng đời 6 tháng**
**Mục tiêu**: Đánh giá hiệu quả tài chính và Tỷ suất Lợi nhuận (ROI) của từng Chiến dịch trong vòng đời 6 tháng của khách hàng.
* **Dòng tiền thực (Real User Charge)**: `userChargeAmount` (tiền túi khách hàng chi ra).
* **Chi phí trợ giá (Subsidy)**: `discountAmount` (tiền ứng dụng tài trợ).
* **Công thức ROI tạm tính**: 
$$\text{ROI} = \frac{\text{userChargeAmount} - \text{discountAmount}}{\text{discountAmount}}$$
* **Vòng đời 6 tháng**: Giới hạn trong 180 ngày kể từ ngày NPU (`days_since_npu <= 180`).

In [6]:
# Tính số ngày kể từ lần giao dịch đầu tiên (NPU)
df['days_since_npu'] = (df['reqDate'] - df['npu_reqDate']).dt.days

# Giới hạn trong vòng đời 6 tháng (180 ngày)
df_6m = df[df['days_since_npu'] <= 180].copy()

# Group theo chiến dịch NPU (Acquisition Campaign)
npu_camp_roi = df_6m.groupby('npu_campaignID').agg(
    total_user_charge=('userChargeAmount', 'sum'),
    total_discount=('discountAmount', 'sum'),
    user_count=('userID', 'nunique')
).reset_index()

# Lọc bỏ campaign 0 và đảm bảo số lượng user thu được >= 10 để tránh nhiễu
npu_camp_roi = npu_camp_roi[(npu_camp_roi['npu_campaignID'] != '0') & (npu_camp_roi['npu_campaignID'] != 0) & (npu_camp_roi['user_count'] >= 10)].copy()
npu_camp_roi['roi'] = (npu_camp_roi['total_user_charge'] - npu_camp_roi['total_discount']) / npu_camp_roi['total_discount']

top_npu_roi = npu_camp_roi.sort_values(by='roi', ascending=False).head(10)


print("Top 10 Chiến dịch NPU có ROI 6 tháng cao nhất:")
display(top_npu_roi)

Top 10 Chiến dịch NPU có ROI 6 tháng cao nhất:


,npu_campaignID,total_user_charge,total_discount,user_count,roi
198,8945,515558897,1659960,35,309.59
170,8901,75694412,248000,29,304.22
138,8612,522917187,2556726,51,203.53
182,8929,354910901,1976000,55,178.61
17,7052,187275696,1091000,63,170.66
110,8384,178111814,1257960,25,140.59
72,7880,436049766,3382967,42,127.90
190,8937,132975176,1047990,24,125.89
181,8928,146705623,1263700,52,115.09
169,8899,26565453,252000,14,104.42


In [7]:
# Phân tích tốc độ hoàn vốn và tăng trưởng ROI theo từng tháng
top_camps_list = top_npu_roi['npu_campaignID'].tolist()[:5]

monthly_roi_data = []
for camp in top_camps_list:
    df_camp = df_6m[df_6m['npu_campaignID'] == camp]
    for month in range(1, 7):
        df_month = df_camp[df_camp['days_since_npu'] <= month * 30]
        charge = df_month['userChargeAmount'].sum()
        discount = df_month['discountAmount'].sum()
        roi = (charge - discount) / discount if discount > 0 else 0
        monthly_roi_data.append({
            'CampaignID': str(camp),
            'Month': month,
            'ROI': roi
        })

df_monthly_roi = pd.DataFrame(monthly_roi_data)

# Hiển thị bảng dữ liệu tăng trưởng ROI
df_pivot_roi = df_monthly_roi.pivot(index='CampaignID', columns='Month', values='ROI')
print("Tăng trưởng lũy kế ROI tạm tính qua 6 tháng của top chiến dịch:")
display(df_pivot_roi)

Tăng trưởng lũy kế ROI tạm tính qua 6 tháng của top chiến dịch:


Month,1,2,3,4,5,6
CampaignID,,,,,,
7052,131.06,153.14,158.05,158.90,162.42,170.66
8612,85.82,109.55,137.09,161.16,190.07,203.53
8901,252.11,294.63,300.80,304.22,304.22,304.22
8929,106.82,160.59,176.12,178.61,178.61,178.61
8945,224.14,289.72,307.31,309.59,309.59,309.59


In [8]:
# Trực quan hóa tốc độ tăng trưởng ROI qua 6 tháng bằng Plotly
fig = px.line(
    df_monthly_roi,
    x='Month',
    y='ROI',
    color='CampaignID',
    markers=True,
    labels={'Month': 'Tháng trong vòng đời người sử dụng', 'ROI': 'Tỷ lệ ROI', 'CampaignID': 'Chiến dịch NPU'},
    title='Tốc độ tăng trưởng ROI qua 6 tháng của các  chiến dịch NPU'
)

# Thêm đường hòa vốn
fig.add_shape(
    type="line",
    x0=1, y0=0, x1=6, y1=0,
    line=dict(color="Red", width=2, dash="dash")
)

fig.add_annotation(
    x=1.5, y=15,
    text="Đường hòa vốn (ROI = 0)",
    showarrow=False,
    font=dict(color="red", size=10)
)

fig.update_traces(
    line=dict(width=3.5),
    marker=dict(size=10),
    hovertemplate='<b>Campaign</b>: %{customdata[0]}<br><b>Tháng</b>: %{x}<br><b>ROI</b>: %{y:.2f}<extra></extra>',
    customdata=np.stack([df_monthly_roi['CampaignID']], axis=-1)
)

fig.update_layout(
    title_x=0.5,
    title_font=dict(size=16, family="Inter, sans-serif", color='#2c3e50'),
    font=dict(family="Inter, sans-serif", color='#2c3e50'),
    plot_bgcolor='#ffffff',
    paper_bgcolor='#ffffff',
    xaxis=dict(
        tickmode='array',
        tickvals=list(range(1, 7)),
        ticktext=[f"Tháng {i}" for i in range(1, 7)],
        gridcolor='rgba(200, 200, 200, 0.15)',
        zerolinecolor='rgba(200, 200, 200, 0.15)'
    ),
    yaxis=dict(
        gridcolor='rgba(200, 200, 200, 0.15)',
        zerolinecolor='rgba(200, 200, 200, 0.15)'
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(t=80, b=40, l=40, r=40),
    height=480
)

fig.show()

**Kết luận**:
   - Các chiến dịch NPU hàng đầu mang lại ROI cực kỳ ấn tượng trong vòng đời 6 tháng. Nổi bật nhất là Chiến dịch `8945` đạt ROI tích lũy là **309.59** và Chiến dịch `8901` đạt ROI là **304.22**.
   - Đáng chú ý, cả hai chiến dịch này đều **đạt điểm hòa vốn và mang lại ROI dương cực nhanh ngay trong Tháng đầu tiên (Month 1)** của vòng đời người dùng (đều đạt ROI > 200). Điều này cho thấy đây là những chiến dịch có thiết kế thông minh, kích thích người dùng bỏ tiền túi rất hiệu quả và bền vững.

## **Phần 3: Hành vi bào khuyến mãi (Deal Hunter Behavior)**
**Mục tiêu**: Phát hiện và đo lường nhóm người dùng chỉ xuất hiện khi có chương trình khuyến mãi (sử dụng voucher/direct discount) và hoàn toàn không phát sinh giao dịch tự nhiên (Organic) trong bất kỳ khoảng thời gian nào.
* **Deal Hunter (Thợ săn deal)**: Nhóm khách hàng có 100% giao dịch gắn liền với `campaignID != 0` (và `promotion_type != 'unknown'`). Họ không đóng góp bất kỳ giao dịch tự nhiên nào.
* **Organic-Prone User (Người dùng trung thành/tự nhiên)**: Khách hàng có phát sinh ít nhất một giao dịch tự nhiên (`campaignID == 0`).

In [9]:
# Phân tích tỷ lệ giao dịch khuyến mãi trên mỗi người dùng
user_promo = df.groupby('userID').agg(
    total_tx=('transID', 'count'),
    promo_tx=('campaignID', lambda x: (x != 0).sum()),
    npu_campaignID=('npu_campaignID', 'first')
).reset_index()

user_promo['pct_promo'] = (user_promo['promo_tx'] / user_promo['total_tx']) * 100

# Phân nhóm người dùng
def segment_user(row):
    if row['pct_promo'] == 100:
        return '1. Deal Hunter (100% Promo)'
    elif row['pct_promo'] >= 50:
        return '2. Promo-Prone (50% - 99% Promo)'
    elif row['pct_promo'] > 0:
        return '3. Organic-Prone (1% - 49% Promo)'
    else:
        return '4. Pure Organic (0% Promo)'

user_promo['Segment'] = user_promo.apply(segment_user, axis=1)

# Thống kê số lượng và tỷ lệ
segment_counts = user_promo['Segment'].value_counts().sort_index().reset_index()
segment_counts.columns = ['Segment', 'User Count']
segment_counts['Percentage (%)'] = (segment_counts['User Count'] / len(user_promo)) * 100

print(f"Tổng số người dùng được phân tích: {len(user_promo):,}")
display(segment_counts)

Tổng số người dùng được phân tích: 33,634


,Segment,User Count,Percentage (%)
0,1. Deal Hunter (100% Promo),13165,39.14
1,2. Promo-Prone (50% - 99% Promo),8749,26.01
2,3. Organic-Prone (1% - 49% Promo),11708,34.81
3,4. Pure Organic (0% Promo),12,0.04


In [10]:
import plotly.express as px

# Bộ màu hiện đại, dịu mắt hơn màu gốc
colors = ['#ff7675', '#fdcb6e', '#55efc4', '#00cec9']
fig = px.pie(
    segment_counts,
    values='User Count',
    names='Segment',
    color_discrete_sequence=colors,
    title='Phân bổ tệp người dùng theo mức độ phụ thuộc khuyến mãi'
)

fig.update_traces(
    # BỎ LABEL TRONG BIỂU ĐỒ: Đặt textinfo='none' để bánh donut sạch sẽ hoàn toàn
    textinfo='none', 
    hole=.5, # Tăng lỗ donut lên một chút nhìn cho thanh thoát
    marker=dict(line=dict(color='#ffffff', width=2)), # Viền trắng phân tách các miếng bánh
    # Tối ưu lại nội dung khi rê chuột vào (Hover) cho gọn gàng
    hovertemplate='<b>%{label}</b><br>Số lượng: %{value:,} user<br>Tỷ lệ: %{percent}<extra></extra>'
)

fig.update_layout(
    title_x=0.5,
    title_font=dict(size=18, family="Arial, sans-serif", color='#2d3436', weight='bold'),
    font=dict(family="Arial, sans-serif", color='#636e72'),
    
    # Cấu hình lại Legend (Chú thích) nằm ngang phía dưới
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.4, # Đẩy xuống dưới một chút để không dính vào bánh
        xanchor="left",
        x=6,
        font=dict(size=14)
    ),
    paper_bgcolor='#ffffff',
    plot_bgcolor='#ffffff',
   # margin=dict(t=80, b=100, l=40, r=40), # Nới rộng margin dưới để chứa legend
    height=550
)

fig.show()

**Kết Luận**:
   - Có tới **39.14%** số lượng người dùng (tương đương 13,165 khách hàng) là các **Deal Hunter thuần túy** (chỉ thực hiện giao dịch khi có khuyến mãi và hoàn toàn biến mất khỏi app khi hết chương trình).
   - Nhóm Promo-Prone (50% - 99% giao dịch khuyến mãi) chiếm **26.01%**.
   - Nhóm khách hàng có hành vi tự nhiên chiếm **34.81%** (gồm Organic-Prone và Pure Organic).
   - **Khuyến nghị**: Tệp Deal Hunter chiếm gần 40% là một tỷ lệ tương đối lớn. Do đó, cần có chiến lược phân lớp khách hàng (customer segmentation) để giảm thiểu khuyến mãi tràn lan cho nhóm này, đồng thời chuyển dịch ngân sách sang nhóm Organic-Prone nhằm tối ưu hóa chi phí marketing (Marketing Spend Optimization).